# Ch 25 — 분류·NER 파인튜닝 (Encoder)

Decoder vs Encoder 차이, 한국어 klue/bert-base 로 도메인 NER (PHONE·MONEY·PRODUCT·CONTRACT) 파인튜닝. IOB 태깅 + seqeval F1 평가.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch25_encoder_ner.ipynb)

In [ ]:
# !pip install -q transformers datasets seqeval

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. NER 작업 정의 — IOB 태깅 (최소 예제)

콜 전사문에서 entity 추출 예시:
```
입력: "휴대폰 010-1234-5678 로 14만원 환불 부탁드립니다"
출력:
  토큰         태그
  휴대폰      O
  010         B-PHONE
  -           I-PHONE
  ...
  14          B-MONEY
  만원        I-MONEY
```

In [ ]:
# Span → IOB 변환 유틸리티

def to_iob(text, entities, tokenizer):
    """char-span 을 token-IOB 로."""
    enc = tokenizer(text, return_offsets_mapping=True)
    offsets = enc.offset_mapping
    labels = ["O"] * len(offsets)
    for ent in entities:
        first = True
        for i, (s, e) in enumerate(offsets):
            if s >= ent["start"] and e <= ent["end"]:
                labels[i] = ("B-" if first else "I-") + ent["label"]
                first = False
    return enc.input_ids, labels

# 예시 확인
tok_test = AutoTokenizer.from_pretrained("klue/bert-base")
text = "휴대폰 010-1234-5678 로 14만원 환불 부탁드립니다"
entities = [
    {"start": 4, "end": 17, "label": "PHONE"},
    {"start": 20, "end": 24, "label": "MONEY"},
]
ids, labels = to_iob(text, entities, tok_test)
tokens = tok_test.convert_ids_to_tokens(ids)
print("token\t\tlabel")
for t, l in zip(tokens, labels):
    print(f"{t:12s}\t{l}")

## 5. 데이터 합성 — 100 문장으로 시작

In [ ]:
# ner_synth.py — 합성 데이터 (Anthropic API 있으면 활성화)
import random, json

# API 없이 간단한 템플릿 기반 합성
PHONES = ["010-1234-5678", "010-9876-5432", "010-1111-2222", "010-3333-4444"]
MONEYS = ["14만원", "50,000원", "3만원", "150만원", "20만원"]
PRODUCTS = ["갤럭시 S25", "아이폰 16", "갤럭시 탭", "맥북 프로"]
CONTRACTS = ["KR-2026-001", "KR-2025-123", "KR-2024-999"]

TEMPLATES = [
    ("휴대폰 {phone} 으로 {money} 환불 부탁드립니다",
     lambda t, p, m: [
         {"start": t.index(p), "end": t.index(p)+len(p), "label": "PHONE"},
         {"start": t.index(m), "end": t.index(m)+len(m), "label": "MONEY"},
     ]),
    ("{product} 계약번호 {contract} 확인 부탁드립니다",
     lambda t, pr, c: [
         {"start": t.index(pr), "end": t.index(pr)+len(pr), "label": "PRODUCT"},
         {"start": t.index(c), "end": t.index(c)+len(c), "label": "CONTRACT"},
     ]),
]

samples = []
for _ in range(100):
    tmpl_fn, ent_fn = random.choice(TEMPLATES)
    if "{phone}" in tmpl_fn:
        p = random.choice(PHONES)
        m = random.choice(MONEYS)
        text = tmpl_fn.format(phone=p, money=m)
        ents = ent_fn(text, p, m)
    else:
        pr = random.choice(PRODUCTS)
        c = random.choice(CONTRACTS)
        text = tmpl_fn.format(product=pr, contract=c)
        ents = ent_fn(text, pr, c)
    samples.append({"text": text, "entities": ents})

with open("ner_train.jsonl", "w") as f:
    for s in samples: f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"합성 완료: {len(samples)} 문장")
print("예시:", samples[0])

## 6. 학습 — transformers Trainer (실전)

In [ ]:
# ner_train.py
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import load_dataset

base = "klue/bert-base"
LABELS = ["O", "B-PHONE", "I-PHONE", "B-MONEY", "I-MONEY",
          "B-PRODUCT", "I-PRODUCT", "B-CONTRACT", "I-CONTRACT"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in id2label.items()}

tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForTokenClassification.from_pretrained(
    base, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

ds = load_dataset("json", data_files="ner_train.jsonl")["train"]

def preprocess(batch):
    enc = tok(batch["text"], truncation=True, max_length=128,
               padding="max_length", return_offsets_mapping=True)
    all_labels = []
    for i, ents in enumerate(batch["entities"]):
        text = batch["text"][i]
        offsets = enc.offset_mapping[i]
        labels = [-100] * len(offsets)  # -100 = ignore in loss
        for j, (s, e) in enumerate(offsets):
            if s == e == 0:
                continue  # special tokens
            tag = "O"
            for ent in ents:
                if s >= ent["start"] and e <= ent["end"]:
                    prefix = "B-" if s == ent["start"] else "I-"
                    tag = prefix + ent["label"]
                    break
            labels[j] = label2id.get(tag, 0)
        all_labels.append(labels)
    enc["labels"] = all_labels
    enc.pop("offset_mapping")
    return enc

ds = ds.map(preprocess, batched=True)
print(f"학습 데이터: {len(ds)} 문장")

In [ ]:
args = TrainingArguments(
    output_dir="ner_out", num_train_epochs=5,
    learning_rate=3e-5, per_device_train_batch_size=16,
    warmup_ratio=0.1, lr_scheduler_type="linear",
    bf16=(device == "cuda"), logging_steps=10,
)
trainer = Trainer(
    model=model, args=args, train_dataset=ds, tokenizer=tok,
    data_collator=DataCollatorForTokenClassification(tok)
)
trainer.train()
trainer.save_model("ner_out/final")

## 7. 추론 + F1 평가

In [ ]:
# ner_eval.py
from transformers import pipeline

ner = pipeline("token-classification", model="ner_out/final",
                aggregation_strategy="simple")  # B-/I- 자동 합침

test_texts = [
    "휴대폰 010-1234-5678로 14만원 환불 부탁드립니다",
    "갤럭시 S25 계약번호 KR-2026-001 확인 부탁드립니다",
]
for t in test_texts:
    res = ner(t)
    print(f"입력: {t}")
    for r in res:
        print(f"  [{r['entity_group']}] {r['word']} (score={r['score']:.2f})")
    print()

## 연습문제

1. 본인 도메인 entity 4개 정의 + 합성 100 페어 + klue/bert-base 학습. F1 측정.
2. KoELECTRA vs klue/bert vs xlm-roberta 같은 데이터로 비교. F1 차이는?
3. 학습 데이터 100 / 500 / 1000 / 5000 으로 학습. F1 곡선.
4. 추론 속도 — 본 책 NER 모델 vs Qwen 2.5-0.5B 에 LoRA 한 NER 비교 (p95 latency).
5. **(생각해볼 것)** ITN 을 "encoder NER + post-processing rule" 으로 구현 가능할까? seq2seq 와 비교했을 때 트레이드오프.

In [ ]:
# 연습 1: 본인 도메인 entity 4개 정의 + 학습


In [ ]:
# 연습 2: KoELECTRA vs klue/bert vs xlm-roberta 비교
# base 후보: "monologg/koelectra-base-v3-discriminator", "klue/bert-base", "xlm-roberta-base"


In [ ]:
# 연습 3: 학습 데이터 크기 변화에 따른 F1 곡선


In [ ]:
# 연습 4: 추론 속도 비교
import time

text = "휴대폰 010-1234-5678로 14만원 환불 부탁드립니다"
n = 100
start = time.perf_counter()
for _ in range(n):
    _ = ner(text)
elapsed = (time.perf_counter() - start) / n * 1000
print(f"NER 평균 추론 시간: {elapsed:.2f} ms/문장")
